# Microsoft Foundry ഉപയോഗിച്ച് ഒരു മോഡൽ ഫൈൻ-ട്യൂൺ ചെയ്യുക

ഈ നോട്ട്‌ബുക്ക് നിലവിലെ [Microsoft Foundry fine-tuning പ്രവൃത്തി പ്രവാഹം](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/fine-tuning?WT.mc_id=academic-105485-koreyst) പിന്ത援ുന്നു. ഇത് നിങ്ങളുടെ Foundry ресурсിന്‍റെ `/openai/v1/` എന്റ്പോയിന്റിൽ കാണുന്ന **OpenAI Python SDK (v1)** ഉപയോഗിക്കുന്നു, അതിനാൽ ഒരേ കോഡ് ക്ലയന്റ് സജ്ജീകരണം മാത്രം മാറി OpenAI പ്ലാറ്റ്‌ഫോമിൻറെ മേൽ പ്രവർത്തിക്കും.

> **ആവശ്യങ്ങൾ**
> - [Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) പ്രോജക്ട്, അതിൽ **Foundry Owner** അവകാശം (fine-tune ചെയ്യാനും ഡിപ്പ്ലോയ്ചെയ്യാനും ആവശ്യമാണ്).
> - `pip install "openai>=1.0" python-dotenv`
> - `AZURE_OPENAI_ENDPOINT` ಮತ್ತು `AZURE_OPENAI_API_KEY` ഉൾക്കൊള്ളുന്ന `.env` ഫയൽ ([കോഴ്‌സ് സജ്ജീകരണ ഗൈഡ് നോക്കുക](./../../../00-course-setup/02-setup-local.md?WT.mc_id=academic-105485-koreyst)).
> - പിന്തുണയുള്ള ഒരു അടിസ്ഥാന മോഡൽ, ഉദാ. `gpt-4.1-mini` ([ഫൈൻ-ട്യൂണബിൾ മോഡലുകളുടെ പട്ടിക കാണുക](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst#fine-tuning-models)).

ഫൈൻ-ട്യൂണിംഗ് നിങ്ങളുടെ ടാസ്ക്കിന് ബന്ധപ്പെട്ട അധിക ഉദാഹരണങ്ങളാൽ ഫൗണ്ടേഷൻ മോഡൽ വീണ്ടും പരിശീലിപ്പിച്ച് മെച്ചപ്പെടുത്തുന്നു. _ഫ്യൂ-ഷോട്ട് ലേണിംഗ്_ , _റിട്രീവൽ-ഓഗ്മെന്റഡ് ജനറേഷൻ_ പോലുള്ള പ്രോമ്പ്റ്റ് എൻജിനീയറിംഗ് സാങ്കേതികവിദ്യകൾ പ്രോമ്പ്റ്റ് ബന്ധപ്പെട്ട ഡേറ്റയാൽ വർദ്ധിപ്പിച്ചാലും മോഡലിന്റെ കോൺടെക്സ്റ്റ് വിൻഡോയിലൂടെ യഥেষ্টമാണ്.

ഫൈൻ-ട്യൂണിംഗിൽ മോഡൽ തന്നെ (കോൺടെക്സ്റ്റ് വിൻഡോയിലിടം കിട്ടുന്നതിൽ കൂടുതൽ ഉദാഹരണങ്ങൾ ഉപയോഗിച്ച്) വീണ്ടും പരിശീലിപ്പിച്ച്, ഇൻഫറൻസിനിടെ ആ ഉദാഹരണങ്ങൾ ആവശ്യമില്ലാത്ത ഒരു _കസ്റ്റം_ വേർഷൻ ഡിപ്പ്ലോയ് ചെയ്യുന്നു. ഇതു ഗുണനിലവാരം മെച്ചപ്പെടുത്തുന്നുവെന്നും, കോൺടെക്സ്റ്റ് വിൻഡോ ഒഴിവാകുന്നതുകൊണ്ടും, പ്രോമ്പ്റ്റുകൾ ലഘൂകരിച്ചതിലൂടെ ചെലവ്, ലാറ്റൻസി കുറയ്ക്കുന്നതിലും സഹായിക്കുന്നു. പിന്നിൽ, Foundry പ്രയോജനപ്രദമായ പരിശീലനത്തിന് **LoRA (low-rank adaptation)** ഉപയോഗിക്കുന്നു.

Foundry മൂന്നു സാങ്കേതികവിദ്യകൾ പിന്തുണയ്ക്കുന്നു: ഈ ട്യൂട്ടോറിയലിൽ ഉപയോഗിച്ചിരിക്കുന്ന **സൂപ്പർവൈസ്ഡ് ഫൈൻ-ട്യൂണിംഗ് (SFT)**, കൂടാതെ **DPO** (പ്രിഫറൻസ് അലിഗ്ന്മെന്റ്) ആയും **RFT** (രീംഫോഴ്‌സ്‌മെന്റ് ഫൈൻ-ട്യൂണിംഗ്) ആയും. പ്രവൃത്തി പ്രവാഹം നാലു ഘട്ടങ്ങളുണ്ട്:

1. നിങ്ങളുടെ പരിശീലന **മറ്റും വെരിഫിക്കേഷൻ** ഡാറ്റ തയ്യാറാക്കി അപ്‌ലോഡ് ചെയ്യുക.
2. fine-tuned മോഡൽ സൃഷ്ടിക്കാൻ ട്രെയ്നിംഗ് ജോബ് നടത്തുക.
3. ജോബ് നിരീക്ഷിച്ച്, മെട്രിക്‌സ് റിവ്യൂ ചെയ്ത് ഒരു ചेक്പോയിന്റ് തിരഞ്ഞെടുക്കുക.
4. fine-tuned മോഡൽ ഡിപ്പ്ലോയ് ചെയ്ത്, ഇൻഫറൻസിനായി ഉപയോഗിക്കുക.

ഈ ട്യൂട്ടോറിയലിൽ, `gpt-4.1-mini` SFT ഉപയോഗിച്ച് fine-tune ചെയ്ത് കാലക്രമരാശി സംബന്ധിച്ച ചോദ്യങ്ങൾക്ക് ലിമെറിക് പോലെ മറുപടി നല്‍കുന്ന ഒരു ചാറ്റ്ബോട്ട് നിർമ്മിക്കുന്നു.

--- 


### ഘട്ടം 1.1: നിങ്ങളുടെ ഡാറ്റാസെറ്റ് തയ്യാറാക്കുക

ഘടകങ്ങളുടെ പിരിയഡിക് പട്ടിക മനസ്സിലാക്കാൻ സഹായിക്കുന്ന ഒരു ചാറ്റ്‌ബോട്ട് നിർമ്മിക്കാം, ലിമെറിക്ക് ഉപയോഗിച്ച് ഒരു ഘടകത്തെക്കുറിച്ച് ചോദ്യങ്ങൾക്ക് উত্তরങ്ങൾ നൽകിക്കൽ. _ഈ_ എളുപ്പമുള്ള ട്യൂട്ടോറിയലിൽ, ഡാറ്റയുടെ പ്രതീക്ഷിക്കുന്ന ഫോർമാറ്റ് കാണിക്കുന്ന ചില സാമ്പിൾ മറുപടികളോടെ ഒരു മോഡൽ ട്രെയിൻ ചെയ്യുന്നതിനായി ഒരു ഡാറ്റാസെറ്റ് സൃഷ്ടിക്കുമെന്ന് മാത്രം. യാഥാർത്ഥ്യത്തിൽ ഉപയോഗിക്കുന്നതിന് നിങ്ങൾക്ക് കൂടുതൽ ഉദാഹരണങ്ങളുള്ള ഒരു ഡാറ്റാസെറ്റ് ഉണ്ടാക്കേണ്ടതുണ്ടാകും. ഒരു തുറന്ന ഡാറ്റാസെറ്റ് (നിങ്ങളുടെ അപേക്ഷ ഡൊമെയ്‌നിന്) ഉണ്ടെങ്കിൽ അത് ഉപയോഗിച്ച് fine-tuning ക്കായി পুনർഫോർമാറ്റുചെയ്യാൻ കഴിയും.

നാം `gpt-4.1-mini` ൽ കേന്ദ്രീകരിക്കുകയും ഒറ്റ-ടേൺ മറുപടി (ചാറ്റ് പൂർത്തീകരണം) അന്വേഷിക്കുകയും ചെയ്യുന്ന സാഹചര്യത്തിൽ, OpenAI ചാറ്റ് പൂർത്തീകരണ ആവശ്യകതകൾ പ്രകടിപ്പിക്കുന്ന [ഈ നിർദ്ദേശിച്ച ഫോർമാറ്റ്](https://platform.openai.com/docs/guides/fine-tuning/preparing-your-dataset?WT.mc_id=academic-105485-koreyst) ഉപയോഗിച്ച് ഉദാഹരണങ്ങൾ സൃഷ്ടിക്കാം. നിങ്ങൾക്ക് മൾട്ടി-ടേൺ സംഭാഷണ ഉള്ളടക്കം പ്രതീക്ഷിക്കുന്നുവെങ്കിൽ, fine-tuning പ്രക്രിയയിൽ ഉപയോഗിക്കേണ്ട (അല്ലെങ്കിൽ വേണ്ട) സന്ദേശങ്ങളെ സൂചിപ്പിക്കാൻ `weight` പാരാമീറ്റർ ഉൾക്കുന്ന [മൾട്ടി-ടേൺ ഉദാഹരണ ഫോർമാറ്റ്](https://platform.openai.com/docs/guides/fine-tuning/multi-turn-chat-examples?WT.mc_id=academic-105485-koreyst) ഉപയോഗിക്കണം.

ഇവിടെ നാം നമ്മുടെ ട്യൂട്ടോറിയലിന് ലളിതമായ ഒറ്റ-ടേൺ ഫോർമാറ്റ് ഉപയോഗിക്കും. ഡാറ്റ [jsonl ഫോർമാറ്റിലാണ്](https://jsonlines.org/?WT.mc_id=academic-105485-koreyst), ഓരോ ലൈനിലും 1 റെക്കോർഡ്, JSON-ഫോർമാട്ട് ഓബ്ജക്റ്റായി പ്രതിനിധാനം ചെയ്യപ്പെടുന്നു. താഴെയുള്ള സ്നിപ്പെറ്റ് 2 റെക്കോർഡുകൾ സാമ്പിളായി കാണിക്കുന്നു - പൂർണ്ണ സാമ്പിൾ സെറ്റ് (10 ഉദാഹരണങ്ങൾ) കാണാൻ [training-data.jsonl](../../../../../18-fine-tuning/python/openai/training-data.jsonl) കാണുക, fine-tuning ട്യൂട്ടോറിയലിന്. **കുറിപ്പ്:** ഓരോ റെക്കോർഡും _ഒരു ലൈനിൽ_ നിർവചിക്കണം (ഫോർമാറ്റ് ചെയ്‌ത JSON ഫയലിൽ സാധാരണയായി കാണുന്ന പോലെ ലൈനുകൾ വിഭജിക്കരുത്)

```json
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Gallium"}, {"role": "assistant", "content": "Gallium, oh gallium, so light - Melts in your hand, oh what a sight - At 86 degrees - Its liquid with ease - And in semiconductors, it's out of sight"}]}
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Hydrogen"}, {"role": "assistant", "content": "Hydrogen, the first in the line - The lightest of all, so divine - It's in water, you see - And in stars, it's the key - The universe's most common sign"}]}
```

യഥാർത്ഥ സാഹചര്യത്തിൽ നല്ല പ്രതിഫലങ്ങൾക്കായി വലിയ ഉദാഹരണങ്ങൾ ഉള്ള സെറ്റ് ആവശ്യമാണ് - fine-tuning ന് വേണ്ടി ഗുണനിലവാരവും സമയം/ചെലവും തമ്മിലുള്ള ആശയവിനിമയമാണ് ഇത്. നാം ചെറിയ ഒരു സെറ്റ് ഉപയോഗിക്കുന്നു, fine-tuning പ്രക്രിയ വേഗത്തിൽ പൂർത്തിയാക്കുന്നതിനായി ഇതു ചെയ്യുന്നു. കൂടുതൽ സങ്കീർണ്ണമായ fine-tuning ട്യൂട്ടോറിയലിനായി [ഈ OpenAI കുക്ക്ബുക്ക് ഉദാഹരണം](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst) കാണുക.


---

### ഘട്ടം 1.2: നിങ്ങളുടെ ഡാറ്റാസെറ്റ് അപ്‌ലോഡ് ചെയ്യുക

Files API (`purpose="fine-tune"`) ഉപയോഗിച്ച് നിങ്ങളുടെ പരിശീലനവും സാധുത ഉറപ്പാക്കുന്നതിനുള്ള ഫയലുകളും Foundry-യിൽ അപ്‌ലോഡ് ചെയ്യുക. ഒരു **സാധുത ഉറപ്പാക്കൽ ഫയൽ** നൽകുന്നത് Foundry-ന് സാധുത നഷ്ടം റിപ്പോർട്ട് ചെയ്യാനാകും, അതിലൂടെ നിങ്ങൾ ഓവർഫിറ്റിംഗ് തിരിച്ചറിയാൻ കഴിയും.

താഴെ കാണുന്ന കോഡ് നിർവഹിക്കുന്നതിന് മുമ്പ്, നിങ്ങൾക്കുണ്ടാകേണ്ടത്:
 - SDK ഇൻസ്റ്റാൾ ചെയ്തിട്ടുണ്ട്: `pip install "openai>=1.0" python-dotenv`
 - `AZURE_OPENAI_ENDPOINT`യും `AZURE_OPENAI_API_KEY`ഉം ഉൾപ്പടെ `.env` ഫയൽ സൃഷ്ടിച്ചിട്ടുണ്ട്

കോഡ്, നിങ്ങളുടെ Foundry ഉറവിടത്തിന്റെ `/openai/v1/` എൻഡ്പോയിന്റിലേക്ക് അടിയന്തരമായ OpenAI ക്ലയന്റിനെ സൃഷ്ടിക്കുകയും, രണ്ടും ഫയലുകളും അപ്‌ലോഡ് ചെയ്ത് അവരുടെ ഐഡികൾ പ്രിന്റ് ചെയ്യുകയും ചെയ്യും.


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Point the OpenAI SDK at your Microsoft Foundry resource's /openai/v1/ endpoint.
# (For the OpenAI platform instead, use: client = OpenAI()  with OPENAI_API_KEY set.)
client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
)

training_response = client.files.create(
    file=open("./training-data.jsonl", "rb"), purpose="fine-tune"
)
validation_response = client.files.create(
    file=open("./validation-data.jsonl", "rb"), purpose="fine-tune"
)

training_file_id = training_response.id
validation_file_id = validation_response.id

print("Training file ID:", training_file_id)
print("Validation file ID:", validation_file_id)


---

### ഘട്ടം 2.1: SDK ഉപയോഗിച്ച് ഫൈൻ-ട്യൂണിംഗ് ജോബ് സൃഷ്ടിക്കുക


In [ ]:
# Create the fine-tuning job.
# trainingType "GlobalStandard" is the recommended tier (lower cost, faster queue).
# Use "Standard" if you need regional data residency, or "Developer" for cheap experiments.
job = client.fine_tuning.jobs.create(
    model="gpt-4.1-mini",              # base model to fine-tune
    training_file=training_file_id,
    validation_file=validation_file_id,
    suffix="elements-limerick",        # helps you identify the resulting model
    seed=105,                          # makes the run reproducible
    extra_body={"trainingType": "GlobalStandard"},
)

job_id = job.id
print("Fine-tuning Job ID:", job_id)
print("Status:", job.status)


---

### ഘട്ടം 2.2: ജോബിന്റെ നില പരിശോധിക്കുക

`client.fine_tuning.jobs` API ഉപയോഗിച്ച് നിങ്ങൾ ചെയ്യാൻ കഴിയുന്ന ചില കാര്യങ്ങൾ ഇവയാണ്:
- `client.fine_tuning.jobs.list(limit=<n>)` - കഴിഞ്ഞ n ഫൈൻ-ട്യൂണിംഗ് ജോലികളിന്റെ പട്ടിക
- `client.fine_tuning.jobs.retrieve(<job_id>)` - ഒരു പ്രത്യേക ജോബിന്റെ വിവരങ്ങൾ നേടുക
- `client.fine_tuning.jobs.cancel(<job_id>)` - ഒരു ജോബ് റദ്ദാക്കുക
- `client.fine_tuning.jobs.list_events(fine_tuning_job_id=<job_id>, limit=<n>)` - ജോബിൽ നിന്നുള്ള സംഭവങ്ങളുടെ പട്ടിക
- `client.fine_tuning.jobs.checkpoints.list(<job_id>)` - ഡിപ്പ്ലോയിംഗിന് ഉപയോഗിക്കാവുന്ന ചെക്ക്‌പോയിന്റുകളുടെ പട്ടിക (ഏതാണ്ട് അവസാന കുറച്ച് എപ്പോക്കുകൾ)

ജോബ് തുടങ്ങുമ്പോൾ, ഫൗണ്ടറി ആദ്യം _പരിശീലന ഫയൽ ശരിയായ രൂപത്തിലാണ് എന്ന് സ്ഥിരീകരിക്കുന്നു_. തുടർന്ന് പരിശീലനം നടക്കുകയും മോഡൽ മാപനശേഷിയുടെയും ഡാറ്റാസെറ്റ് വലുപ്പത്തിന്റെയും അടിസ്ഥാനത്തിൽ training ഇപ്പോൾ മിനിറ്റുകൾ മുതൽ മണിക്കൂറുകൾ വരെ വേണ്ടിവരാം.


In [ ]:
# List the last 10 fine-tuning jobs
client.fine_tuning.jobs.list(limit=10)

# Retrieve the current state of your fine-tuning job
client.fine_tuning.jobs.retrieve(job_id)

# List up to 10 events from the job
client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10)


In [ ]:
# Track the job status until it is complete
response = client.fine_tuning.jobs.retrieve(job_id)

print("Job ID:", response.id)
print("Status:", response.status)
print("Trained Tokens:", response.trained_tokens)


---

### ഘട്ടം 2.3: പുരോഗതി നിരീക്ഷിക്കാൻ ഇവന്റ്‌സ് ട്രാക്ക് ചെയ്യുക


In [ ]:
# Track progress in a more granular way by checking events.
# Re-run this cell until you see "The job has successfully completed".
response = client.fine_tuning.jobs.list_events(job_id)

events = response.data
events.reverse()

for event in events:
    print(event.message)


In [ ]:
# When the job finishes, the last few epochs are available as deployable checkpoints.
# Best practice: don't blindly deploy the final epoch - review the checkpoints and pick
# the one that generalizes best (watch train_loss vs. valid_loss and token accuracy).
checkpoints = client.fine_tuning.jobs.checkpoints.list(job_id)
for cp in checkpoints.data:
    print(cp.fine_tuned_model_checkpoint, "| step:", cp.step_number)


### ഘട്ടം 2.4: ഫൗണ്ട്രി പോർട്ടലിൽ സ്റ്റാറ്റസ്, മെറ്റ്രിക്‌സ്, ചെക്ക്പോയിന്റുകൾ പരിശോധിക്കുക


നിങ്ങൾക്ക് [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) ൽ **Build > Fine-tuning** എന്ന അദ്ധ്യായത്തിലൂടെ ജോബ് ട്രാക്ക് ചെയ്യാനും കഴിയും. നിങ്ങളുടെ ജോബ് തെരഞ്ഞെടുക്കുക അതിന്റെ നില, പരിശീലന ഇവന്റുകൾ, ഹൈപ്പർപരാമീറ്ററുകൾ, മെട്രിക്സ് എന്നിവ കാണാൻ. ഈ മെട്രിക്സ് ശ്രദ്ധിക്കുക:

- `train_loss` and `full_valid_loss` - സമയക്രമത്തിൽ കുറഞ്ഞിരിക്കണം.
- `train_mean_token_accuracy` and `full_valid_mean_token_accuracy` - വർദ്ധിക്കണം.

പരിശീലനവും സമ്മതീകരണവും വളക്കൽവുമോർക്കുന്നേക്കാമെന്ന് തോന്നിയാൽ, നിങ്ങൾ അധികമായി ഓവർഫിറ്റിംഗ് ചെയ്തേക്കാം - കുറച്ചുകാലം(epoch) പരീക്ഷിക്കുക അല്ലെങ്കിൽ ചെറിയ ലേണിംഗ് റേറ്റ് മൾട്ടിപ്പ്ലയറി ഉപയോഗിക്കുക. താഴെ കാണിക്കുന്ന ചിത്രം നിങ്ങൾ കാണാനുള്ള സ്‌ഥിതി, സന്ദേശം, മെട്രിക്‌സ് പാനലുകളുടെ മാതൃകയാണ് (വാസ്തവ UI സേവന ദാതാവിന്റെ ബേസ് വ്യത്യാസപ്പെടും).

![Fine-tuning job status](../../../../../translated_images/ml/fine-tuned-model-status.563271727bf7bfba.webp)


താഴേക്ക് കൂടുതൽ സ്‌ക്രോൾ ചെയ്ത് കാഴ്ചാ ഡാഷ്ബോർഡിൽ പ്രവർത്തിയുടെ സന്ദേശങ്ങളും മെട്രിക്കുകളും നിങ്ങൾക്ക് കാണാൻ കഴിയും, ചുവടെ കാണിക്കുപോലെ:

| സന്ദേശങ്ങൾ | മെട്രിക്കുകൾ |
|:---|:---|
| ![Messages](../../../../../translated_images/ml/fine-tuned-messages-panel.4ed0c2da5ea1313b.webp) |  ![Metrics](../../../../../translated_images/ml/fine-tuned-metrics-panel.700d7e4995a65229.webp)|


---

### ഘട്ടം 3.1: നല്ലപടി പരിശീലിപ്പിച്ച മോഡൽ ഐഡി വാങ്ങുക

ജോബ് വിജയിച്ചാൽ, `response.fine_tuned_model` നിങ്ങളുടെ റ്ററെയിൽ മോഡലിന്റെ ഐഡി സൂക്ഷിക്കും (ഉദാഹരണത്തിന്, `gpt-4.1-mini-2025-04-14.ft-...`). Azure-യിൽ നിങ്ങൾ അത് **ഡിപ്പ്ലോയ്** ചെയ്യാൻ മാത്രമേ കഴിയൂ അതിന് ശേഷം മാത്രമേ അത് വിളിക്കാൻ സാധിക്കൂ.


In [ ]:
# Retrieve the id of the fine-tuned model once the job has succeeded
response = client.fine_tuning.jobs.retrieve(job_id)
fine_tuned_model_id = response.fine_tuned_model
print("Fine-tuned Model ID:", fine_tuned_model_id)


### ഘട്ടം 3.2: ഫൈൻ-ട്യൂൺ ചെയ്‌ത മോഡൽ വിന്യസിക്കുക

OpenAI പ്ലാറ്റ്ഫോമിന് വ്യത്യസ്തമായി (അവിടെ നിങ്ങൾക്ക് ഫൈൻ-ട്യൂൺ ചെയ്‌ത മോഡൽ ഐഡി നേരിട്ട് വിളിക്കാം), Microsoft Foundry ആദ്യം മോഡൽ **വിന്യസിക്കുക** എന്നത് ആവശ്യമാണ്. ഏറ്റവും എളുപ്പവഴി [Foundry പോർട്ടൽ](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) ഉപയോഗിക്കുക: **Build > Fine-tuning**-ലേക്ക് പോവുക, നിങ്ങൾ പൂർത്തിയാക്കിയ ജോബ് തിരഞ്ഞെടുക്കുക, ശേഷം **Deploy** തിരഞ്ഞെടുക്കുക. വിന്യസനത്തിന് ഒരു പേര് നൽകുക (ഉദാഹരണത്തിന്, `elements-limerick`) - ആ വിന്യസനപേര് ആണ് നിങ്ങൾ കോഡിൽ വിളിക്കുന്നതിന്റെ പേരും.

നിങ്ങൾ കൺട്രോൾ-പ്ലെയിൻ REST/ARM API ഉപയോഗിച്ച് പ്രോഗ്രാമാറ്റിക്കായി വിന്യസിക്കാനും കഴിയും; [വിന്യസന മാർഗ്ഗനിർദ്ദേശം](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/fine-tuning-deploy?WT.mc_id=academic-105485-koreyst) കാണുക. വിന്യസിച്ച കസ്റ്റം മോഡലിന് മണിക്കൂറു അടിസ്ഥാനത്തിൽ ബിൽ ചെയ്യുന്നു, കൂടാതെ 15 ദിവസത്തെ ഇടവേളയോടെ അനുപയോഗമായ വിന്യസനം നീക്കം ചെയ്യും എന്നത് ഓർക്കുക.


In [ ]:
# Once deployed, call your fine-tuned model by its DEPLOYMENT name via the Responses API.
# (On the OpenAI platform you'd pass fine_tuned_model_id directly instead.)
deployment_name = "elements-limerick"  # the name you gave the deployment in Foundry

completion = client.responses.create(
    model=deployment_name,
    input=[
        {"role": "system", "content": "You are Elle, a factual chatbot that answers questions about elements in the periodic table with a limerick"},
        {"role": "user", "content": "Tell me about Strontium"},
    ],
    store=False,
)
print(completion.output_text)


---

### ഘട്ടം 3.3: ഫൈൻ-ട്യൂൺ ചെയ്ത മോഡല്‍ ഫൗണ്ട്രി പ്ലേ ഗ്രൗണ്ടില്‍ പരീക്ഷിക്കുക

നിങ്ങളുടെ ഫൈൻ-ട്യൂൺ ചെയ്ത ഡിപ്ലോയ്മെന്റ് മോഡൽ മോഡൽ ഡ്രോപ്ഡൗണിൽ നിന്ന് തിരഞ്ഞെടുത്ത് ഒരു പ്രോമ്പ്റ്റ് പരീക്ഷിക്കാൻ നിങ്ങൾക്ക് [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) **Playground** -ഉം ഉപയോഗിക്കാമെന്ന് അറിയാം. നിങ്ങൾ പരിശീലിപ്പിച്ച **അതിനേകമായി സിസ്റ്റം സന്ദേശം** ഉപയോഗിക്കുക; വ്യത്യസ്തമായ സിസ്റ്റം സന്ദേശം പെരുമാറ്റം മാറ്റാൻ കഴിയും.

> **സൂചനം:** ഫൈൻ-ട്യൂൺ ചെയ്ത മോഡലിനെ അടിസ്ഥാന `gpt-4.1-mini`മോടിനെ വക്കിരി നിരീക്ഷിക്കുക. നിങ്ങളുടെ ഉദാഹരണങ്ങളിൽ നിന്നുള്ള ലിമെരിക്ക് ഫോർമാറ്റിൽ ഫൈൻ-ട്യൂൺ ചെയ്ത മോഡൽ എങ്ങനെ ഉത്തരം നൽകുന്നു എങ്കിൽ മാത്രം അടിസ്ഥാന മോഡൽ സിസ്റ്റം പ്രോമ്പ്റ്റിന്റെ അനുസരണം മാത്രമേ നടത്തൂ.

**ഇത് ഒരുപാട് ലളിതമായ ഒരു ഉദാഹരണമാണ് പ്രക്രിയ കാണിക്കാൻ, യാഥാർത്ഥ്യ ഡാറ്റ സെറ്റ് അല്ല.** പ്രൊഡക്ഷനിൽ നിങ്ങൾ യഥാർത്ഥ ഡാറ്റയിൽ ഫൈൻ-ട്യൂൺ ചെയ്യും (ഉദാഹരണത്തിന്, ഉപഭോക്തൃ സേവനത്തിനുവേണ്ടി ഒരു ഉൽപ്പന്ന കാറ്റലോഗ്), അവിടെ ഗുണമേന്മയുടെ വ്യത്യാസം വളരെ വ്യക്തമാകും - പ്രോമ്പ്റ്റ് എഞ്ചിനീയറിംഗിലൂടെ മാത്രം ആ ഗുണമേന്മ കൈവരിക്കാൻ ഒരുപാട് കൂടുതൽ ടോക്കണുകൾ വിളിക്കേണ്ടി വരും. കൂടുതൽ സമഗ്രമായ ഒരു ഉദാഹരണത്തിനായി, [OpenAI Cookbook fine-tuning guide](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst)യും [Foundry fine-tuning tutorial](https://learn.microsoft.com/azure/ai-foundry/openai/tutorials/fine-tune?WT.mc_id=academic-105485-koreyst)ഉം കാണുക.

---


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
